In [2]:
# Imports
# TODO: change if necessary to accomodate change in data used
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from dataclasses import dataclass, asdict
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [3]:
# Code to obtain activation of network
def get_activation(name):
    activations = {
        "relu": nn.ReLU,
        "gelu": nn.GELU,
        "silu": nn.SiLU,
        "tanh": nn.Tanh,
        "identity": nn.Identity,
    }
    name = name.lower()
    if name not in activations:
        raise ValueError(f"Unknown activation {name}")
    return activations[name]()

In [4]:
# Expert metadata dataclass
@dataclass
class ExpertMetadata:
    expert_id: int
    family: str
    architecture: str
    depth: int
    width: int
    activation: str
    geometry: str = "euclidean"
    num_parameters: int = 0

In [5]:
# Base expert
class BaseExpert(nn.Module):
    """
    Parent class of all experts
    Each expert has common input/output interface, exposes architectural metadata and
    can be analyzed by router's logging system
    TODO: Use this in a later step for the hyperbolic neural net subclass
    """

    def __init__(
        self,
        input_dim=256,
        output_dim=256,
        expert_id=-1,
        family="unknown",
        architecture="unknown",
        depth=0,
        width=0,
        activation="unknown",
        geometry="euclidean",
    ):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.metadata = ExpertMetadata(
            expert_id=expert_id,
            family=family,
            architecture=architecture,
            depth=depth,
            width=width,
            activation=activation,
            geometry=geometry,
        )

    def forward(self, x):
        raise NotImplementedError(
        )

    def update_parameter_count(self):
        """Update number of trainable and non-trainable parameters."""
        self.metadata.num_parameters = sum(
            p.numel() for p in self.parameters()
        )

    def get_metadata(self):
        """Return a copy of the expert's metadata as a dictionary."""
        self.update_parameter_count()
        return asdict(self.metadata)

In [6]:
# MLPExpert
class MLPExpert(BaseExpert):
    """
    Subclass of BaseExpert, default activation is gelu
    """
    def __init__(
        self,
        input_dim=256,
        output_dim=256,
        hidden_dims=(512,),
        activation="gelu",
        residual=False,
        expert_id=-1,
        family="mlp",
        geometry="euclidean",
    ):
        hidden_dims = tuple(hidden_dims)
        # Determine metadata from the actual architecture.
        dims = [input_dim] + list(hidden_dims) + [output_dim]

        architecture = " -> ".join(map(str, dims))

        if residual:
            architecture += " + residual"

        super().__init__(
            input_dim=input_dim,
            output_dim=output_dim,
            expert_id=expert_id,
            family=family,
            architecture=architecture,
            depth=len(dims) - 1,
            width=max(hidden_dims, default=0),
            activation=activation,
            geometry=geometry,
        )
        self.hidden_dims = hidden_dims
        self.activation_name = activation
        self.residual = residual
        if residual and input_dim != output_dim:
            raise ValueError(
                f"res conns req input_dim == output_dim, input dim is {self.input_dim} and output dim is {self.output_dim}"
            )
        
        layers = []

        for i in range(len(dims) - 1):
            layers.append(
                nn.Linear(dims[i], dims[i + 1])
            )
            # Activation after all hidden layers
            if i < len(dims) - 2:
                layers.append(get_activation(activation))
        self.network = nn.Sequential(*layers)
        self.update_parameter_count()

    def forward(self, x):
        output = self.network(x)
        if self.residual:
            output = output + x
        return output

In [7]:
# GeGLU, different because it's fundamentally different from the others
class GEGLUExpert(BaseExpert):
    """
    Gated GELU expert (also subclass of BaseExpert)
    """
    def __init__(
        self,
        input_dim=256,
        hidden_dim=512,
        output_dim=256,
        expert_id=-1,
        geometry="euclidean",
    ):
        super().__init__(
            input_dim=input_dim,
            output_dim=output_dim,
            expert_id=expert_id,
            family="gated",
            architecture=(
                f"{input_dim} -> GEGLU({hidden_dim}) -> {output_dim}"
            ),
            depth=2,
            width=hidden_dim,
            activation="gelu_gate",
            geometry=geometry,
        )
        self.hidden_dim = hidden_dim
        self.input_projection = nn.Linear(
            input_dim,
            hidden_dim * 2,
        )
        self.output_projection = nn.Linear(
            hidden_dim,
            output_dim,
        )
        self.update_parameter_count()

    def forward(self, x):
        value, gate = self.input_projection(x).chunk(
            2, dim=-1
        )
        hidden = value * F.gelu(gate)
        return self.output_projection(hidden)

In [8]:
# Output record of expert metadata
def log_routing_event(experts, probabilities, selected_id):
    expert = experts[selected_id]
    metadata = expert.get_metadata()

    record = {
        **metadata,
        "routing_probability": probabilities[
            selected_id
        ].detach().item(),
    }
    return record

In [9]:
# Expert "factory"
'''Automating creation of experts (because I'm a lazy bum)
Note: default is 256 for input and output'''
# TODO: figure out how to make experts customizable instead of splitting into shallow, wide, etc after
# determining how each fit together to see if I can take that next step (and without needing iteration
# so time complexity is O(1)) and for full range of options, current approach is naive

""" 
    def __init__(
        self,
        input_dim=256,
        output_dim=256,
        expert_id=-1,
        family="unknown",
        architecture="unknown",
        depth=0,
        width=0,
        activation="unknown",   
        geometry="euclidean",
    ):
"""

# TODO: add more inputs to make_expert
def make_expert(
    expert_id,
    family,
    input_dim=256,
    output_dim=256,
    geometry="euclidean",
):
    """Create an expert from a predefined architectural family."""

    # Shallow-wide
    if family == "shallow_wide":
        return MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(1024,),
            activation="gelu",
            expert_id=expert_id,
            family=family,
            geometry=geometry,
        )

    # Shallow-medium
    elif family == "shallow_medium":
        return MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(512,),
            activation="gelu",
            expert_id=expert_id,
            family=family,
            geometry=geometry,
        )

    # Deep-narrow
    elif family == "deep_narrow":
        return MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(256, 256, 256),
            activation="gelu",
            expert_id=expert_id,
            family=family,
            geometry=geometry,
        )

    # Very deep and narrow
    elif family == "very_deep":
        return MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(192, 192, 192, 192, 192),
            activation="gelu",
            expert_id=expert_id,
            family=family,
            geometry=geometry,
        )

    # Activation function variants
    elif family in ("relu", "gelu", "silu", "tanh"):
        return MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(512,),
            activation=family,
            expert_id=expert_id,
            family=family,
            geometry=geometry,
        )

    # Gated linear unit with GELU gating
    elif family == "gated":
        return GEGLUExpert(
            input_dim=input_dim,
            hidden_dim=512,
            output_dim=output_dim,
            expert_id=expert_id,
            geometry=geometry,
        )

    # Residual MLP
    elif family == "residual":
        return MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(512,),
            activation="gelu",
            residual=True,
            expert_id=expert_id,
            family=family,
            geometry=geometry,
        )

    # Bottleneck MLP
    elif family == "bottleneck":
        return MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(64,),
            activation="gelu",
            expert_id=expert_id,
            family=family,
            geometry=geometry,
        )

    # Expansion MLP
    elif family == "expansion":
        return MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(1024,),
            activation="gelu",
            expert_id=expert_id,
            family=family,
            geometry=geometry,
        )

    else:
        raise ValueError(
            f"Unknown expert family {family}. "
        )


In [ ]:
# "Free" labour

families = [
    "shallow_wide",
    "shallow_medium",
    "deep_narrow",
    "very_deep",
    "relu",
    "gelu",
    "silu",
    "tanh",
    "gated",
    "residual",
    "bottleneck",
    "expansion",
]

# Constructing dynamic decision graph
"""experts = nn.ModuleDict(
    {
        f"Expert ID: {i}": make_expert(
            expert_id=i,
            family=families[i % len(families)]
        )
        for i in range(len(families) * 5) # NOTE: 5 for each family right now
    }
)""" # TODO: use after sanity check



ID: 0, family: shallow_wide, shape: (32, 256), depth: 2, width: 1024, parameters: 525,568
ID: 1, family: shallow_medium, shape: (32, 256), depth: 2, width: 512, parameters: 262,912
ID: 2, family: deep_narrow, shape: (32, 256), depth: 4, width: 256, parameters: 263,168
ID: 3, family: very_deep, shape: (32, 256), depth: 6, width: 192, parameters: 246,976
ID: 4, family: relu, shape: (32, 256), depth: 2, width: 512, parameters: 262,912
ID: 5, family: gelu, shape: (32, 256), depth: 2, width: 512, parameters: 262,912
ID: 6, family: silu, shape: (32, 256), depth: 2, width: 512, parameters: 262,912
ID: 7, family: tanh, shape: (32, 256), depth: 2, width: 512, parameters: 262,912
ID: 8, family: gated, shape: (32, 256), depth: 2, width: 512, parameters: 394,496
ID: 9, family: residual, shape: (32, 256), depth: 2, width: 512, parameters: 262,912
ID: 10, family: bottleneck, shape: (32, 256), depth: 2, width: 64, parameters: 33,088
ID: 11, family: expansion, shape: (32, 256), depth: 2, width: 1024, 

In [ ]:
# Sanity check code
experts = nn.ModuleList(
    [
        make_expert(
            expert_id=i,
            family=families[i % len(families)]
        )
        for i in range(len(families) * 5)
    ]
)

x = torch.randn(32, 256)
for expert in experts:
    output = expert(x)
    metadata = expert.get_metadata()
    print(
        f"ID: {metadata['expert_id']}, "
        f"family: {metadata['family']}, "
        f"shape: {tuple(output.shape)}, "
        f"depth: {metadata['depth']}, "
        f"width: {metadata['width']}, "
        f"parameters: {metadata['num_parameters']:,}"
    )

In [ ]:

# Device stuff
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Using:", device)

In [ ]:
# Training
# Note that currently using MNIST dataset that's not complicated for classification
# TODO: use word data instead so change everything (I want to see if this can help capture semantics)

# Data handling (convert then normalize)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.1307,),
        (0.3081,)
    )
])

# Datasets
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)


In [ ]:
# Model
# TODO: changed format of architecture so change this too
model = DynamicDecisionGraph(
    input_dim=28 * 28,
    state_dim=128,
    hidden_dim=128,
    num_classes=10,
    num_experts=8,
    collection=collection,
    max_steps=6
).to(device)

# Adam optimizer selected for its in-built regularizationa and performance
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3 # Current tentative
)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Training
epochs = 100 # Note to self: modify LR as epochs increase, don't run from start

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        images = images.view(
            images.size(0),
            -1
        )

        # Forward pass
        logits = model(images)
        # Loss
        loss = criterion(
            logits,
            labels
        )

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Stats
        total_loss += loss.item()
        predictions = logits.argmax(dim=1)
        correct += (
            predictions == labels
        ).sum().item()
        total += labels.size(0)

    # Overall stats
    accuracy = correct / total
    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"| Loss: {total_loss / len(train_loader):.4f} "
        f"| Accuracy: {accuracy:.4f}"
    )